# V18 – Kryptografie Teil 2: Theorie

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- die drei zentralen Eigenschaften kryptografischer Hash-Funktionen benennen (**deterministisch**, **einweg**, **Lawineneffekt**) und jede davon an einem Beispiel erklären,
- erklären, warum Passwörter niemals im Klartext in einer Datenbank liegen dürfen und wie Hash-Funktionen dieses Problem lösen,
- die Grundidee eines **Wörterbuch-Angriffs** beschreiben und mit dem Konzept **Salt** die wichtigste Gegenmaßnahme erläutern,
- die Sicherheitsunterschiede zwischen MD5, SHA-1 und SHA-256 einordnen,
- das Prinzip einer **digitalen Signatur** als Kombination aus Hash und asymmetrischer Verschlüsselung skizzieren und am Beispiel eines Firmware-Updates beschreiben, wann ein Roboterarm eine Signatur prüft.

## ⏱️ Zeitbudget
Ca. 30 Minuten für die reine Theorie.

## 🧭 TL;DR
Eine **Hash-Funktion** erzeugt aus beliebig großen Eingaben einen festen, scheinbar zufälligen Kurz-Output. Gute Hash-Funktionen sind **einweg** (der Weg zurück ist praktisch unmöglich) und reagieren auf die kleinste Änderung der Eingabe mit einem komplett anderen Ergebnis (**Lawineneffekt**). Passwörter werden deshalb gehasht gespeichert. Angreifer probieren dagegen mit Wortlisten sehr viele Hash-Berechnungen durch – die Gegenmaßnahme heißt **Salt**. Wird der Hash zusätzlich mit einem privaten Schlüssel verschlüsselt, entsteht eine **digitale Signatur**, mit der ein Empfänger die Echtheit einer Datei überprüfen kann.

## 📦 Voraussetzungen
- V17 (Caesar-Chiffre, symmetrische vs. asymmetrische Verschlüsselung).
- `00_python_recap.ipynb`.

## 1. Was ist eine Hash-Funktion?

Stell dir einen digitalen **Fingerabdruck** vor. Egal wie groß ein Mensch ist – sein Fingerabdruck ist immer gleich klein und trotzdem praktisch eindeutig. Genau das leistet eine **Hash-Funktion** mit beliebigen Daten: Sie nimmt eine Eingabe beliebiger Länge – ein einzelnes Wort, einen ganzen Roman oder eine mehrere Gigabyte große Firmware-Datei – und erzeugt daraus einen **kurzen, festen** Wert, den **Hash** oder **Hashwert**. Typische Längen sind 128 Bit (MD5), 160 Bit (SHA-1) oder 256 Bit (SHA-256). Dargestellt werden diese Bits meist als **Hexadezimal-Zeichenkette**.

> [!NOTE]
> Eine **Hash-Funktion** ist eine mathematische Abbildung von einer beliebig langen Eingabe auf einen Ausgabewert fester Länge. Eine **kryptografische** Hash-Funktion erfüllt darüber hinaus zusätzliche Sicherheitseigenschaften, auf die wir gleich im Detail eingehen.

Anders als die Caesar-Chiffre aus V17 ist eine Hash-Funktion **kein Verschlüsselungsverfahren**. Bei einer Verschlüsselung willst du die ursprüngliche Nachricht später wieder entschlüsseln können. Bei einer Hash-Funktion ist das Gegenteil das Ziel: Aus dem Hash soll sich die Eingabe **nicht** wiederherstellen lassen. Eine Hash-Funktion wirft also absichtlich Information weg.

Die folgende Skizze zeigt die drei Kern-Eigenschaften einer guten kryptografischen Hash-Funktion auf einen Blick.

In [1]:
from IPython.display import Markdown, display
with open("diagramme/01_hash_eigenschaften.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
flowchart TD
    A["Beliebige Eingabe<br/>(String, Datei, GB-gross)"] --> B{{"Hash-Funktion<br/>(z.B. SHA-256)"}}
    B --> C["Fester Output<br/>256 Bit = 64 Hex-Zeichen"]
    C --> D["Deterministisch:<br/>gleiche Eingabe &rarr; gleicher Hash"]
    C --> E["Einweg:<br/>Hash &rarr; Eingabe nicht moeglich"]
    C --> F["Lawineneffekt:<br/>1 Bit Aenderung &rarr; komplett anderer Hash"]

```

## 2. Die drei zentralen Eigenschaften

### 2.1 Deterministisch

Eine Hash-Funktion liefert für dieselbe Eingabe **immer** denselben Hash-Wert, egal wann und auf welchem Rechner sie ausgeführt wird. Diese Eigenschaft klingt banal, ist aber die Grundlage jeder Anwendung: Nur weil der Hash reproduzierbar ist, kann ich zwei Dateien vergleichen, indem ich ihre Hashes vergleiche.

### 2.2 Einweg-Eigenschaft

Gegeben der Hash-Wert `h`, ist es **praktisch unmöglich**, eine Eingabe `m` zu finden, deren Hash genau `h` ergibt. Formal spricht man von **Preimage Resistance**. „Praktisch unmöglich" heißt: Mit den schnellsten heute verfügbaren Computern würde eine Brute-Force-Suche über alle denkbaren Eingaben länger dauern als das Universum existiert. Diese Asymmetrie ist der Kern: In die eine Richtung ist die Berechnung billig, in die andere Richtung astronomisch teuer.

> 💡 **Weiterführend**: Es gibt drei verwandte Sicherheitsbegriffe – *Preimage Resistance* (zu gegebenem Hash keinen Input finden), *Second Preimage Resistance* (zu gegebenem Input keinen zweiten Input mit gleichem Hash finden) und *Collision Resistance* (gar kein Paar mit gleichem Hash finden). Für Passwort-Hashing reicht die erste Eigenschaft, für digitale Signaturen brauchst du die dritte.

### 2.3 Lawineneffekt

Selbst eine **winzige Änderung** an der Eingabe – ein umgedrehtes Bit, ein geändertes Komma, ein zusätzliches Leerzeichen – führt dazu, dass sich etwa die Hälfte aller Bits im Hash-Wert ändert. Der Hash sieht danach aus wie ein komplett anderer Zufallswert. Dank des Lawineneffekts erkennst du selbst winzige Manipulationen sofort – ideal für Integritätsprüfungen.

Das folgende Beispiel zeigt den Effekt. Die beiden Strings unterscheiden sich nur in einem Zeichen, die Hashes sind trotzdem völlig unterschiedlich.

In [2]:
import hashlib
print(hashlib.sha256(b"Maschine").hexdigest())
print(hashlib.sha256(b"maschine").hexdigest())  # nur ein Buchstabe anders!

54060135f7b6002ab07a139a9a1b23e9f85c4f4e95ff3f92ac2dd7c006cf4d36
377386820f8746b7313af0a3b1db5e087732df708d3026fb140b5a86fd977745


## 3. Die wichtigsten Hash-Algorithmen im Überblick

Im Laufe der letzten 30 Jahre hat die Kryptografie-Community mehrere Hash-Funktionen entwickelt und wieder verworfen. Der Grund: Sobald effiziente Angriffe bekannt werden, gilt ein Algorithmus als **gebrochen** und darf nicht mehr für Sicherheits-kritische Zwecke verwendet werden. Die folgende Tabelle gibt einen Überblick.

| Algorithmus | Output-Länge | Status | Verwendung |
|---|---|---|---|
| **MD5** | 128 Bit (32 Hex) | ❌ gebrochen seit 2004 | nur noch für Checksummen, **nicht** sicherheitskritisch |
| **SHA-1** | 160 Bit (40 Hex) | ❌ gebrochen seit 2017 | Legacy-Systeme, wird ausgemustert |
| **SHA-256** | 256 Bit (64 Hex) | ✅ sicher | Industrie-Standard, z. B. Bitcoin, TLS, Git |
| **SHA-3** | variabel | ✅ sicher | modernes Alternativ-Design (seit 2015) |
| **bcrypt / Argon2** | variabel | ✅ sicher | speziell für **Passwort**-Hashing |

### 3.1 MD5 – der Klassiker, heute unsicher

**MD5** wurde 1991 von Ron Rivest entwickelt und war jahrelang der De-facto-Standard. 2004 zeigte ein chinesisches Forscher-Team, wie man **Kollisionen** effizient erzeugen kann, also zwei verschiedene Eingaben mit demselben MD5-Hash. Für Passwörter und digitale Signaturen ist MD5 heute tabu. Als schnelle **Checksumme** gegen zufällige Übertragungsfehler (nicht gegen böswillige Manipulation) ist er aber noch in Gebrauch – zum Beispiel wenn eine Linux-Distribution einen `.iso`-Download mit MD5-Prüfsumme anbietet.

### 3.2 SHA-1 – ebenfalls unsicher

**SHA-1** wurde 1995 von der NSA entworfen und lange als Nachfolger von MD5 eingesetzt, etwa in den Versionskontroll-Systemen Git oder in alten TLS-Zertifikaten. 2017 veröffentlichte Google den **SHAttered**-Angriff und zeigte zwei unterschiedliche PDF-Dateien mit identischem SHA-1-Hash. Seitdem gilt auch SHA-1 als gebrochen und wird in neuen Systemen nicht mehr eingesetzt.

### 3.3 SHA-256 – der heutige Standard

**SHA-256** gehört zur SHA-2-Familie und erzeugt einen 256 Bit langen Hash. Es gilt nach aktuellem Stand als sicher und wird praktisch überall eingesetzt: in TLS-Zertifikaten, bei Bitcoin-Transaktionen, in Docker-Image-IDs, in Git-Commit-Hashes (neue Repositories) und eben auch in digitalen Signaturen von Firmware-Updates. Wenn du heute eine Hash-Funktion auswählen musst und kein Passwort-Hashing betreibst, ist SHA-256 meistens die richtige Wahl.

## 4. Warum Passwörter niemals im Klartext gespeichert werden

Stell dir vor, ein Online-Shop speichert alle Kunden-Passwörter im Klartext in einer Datenbank-Tabelle `users(email, passwort)`. Wird dieser Shop gehackt und die Datenbank kopiert – das passiert regelmäßig und nennt sich **Database Dump** – hat der Angreifer auf einen Schlag Zugriff auf alle Passwörter. Da viele Menschen Passwörter **wiederverwenden**, kann er sich anschließend bei E-Mail-Konten, Online-Banking und Social-Media-Profilen derselben Nutzer einloggen.

Die saubere Lösung: Der Server speichert niemals das Passwort selbst, sondern nur dessen **Hash**. Bei jedem Login rechnet der Server den Hash der eingegebenen Eingabe und vergleicht ihn mit dem gespeicherten Hash. Stimmt beides überein, ist die Authentifizierung erfolgreich. Wird die Datenbank gestohlen, hat der Angreifer nur die Hashes – und die Einweg-Eigenschaft macht das Zurückrechnen auf das Klartext-Passwort praktisch unmöglich.

### Login-Ablauf im Detail

In [3]:
from IPython.display import Markdown, display
with open("diagramme/02_login_flow.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
sequenceDiagram
    participant U as Nutzer
    participant S as Server
    participant DB as Datenbank
    U->>S: Login mit Passwort "geheim123"
    S->>S: hash = sha256("geheim123")
    S->>DB: Lade gespeicherten Hash fuer Nutzer
    DB-->>S: gespeicherter_hash
    alt hash == gespeicherter_hash
        S-->>U: Login erfolgreich
    else
        S-->>U: Zugriff verweigert
    end
    Note over DB: Nur Hashes werden gespeichert,<br/>niemals Klartext-Passwoerter.

```

Die Reihenfolge ist wichtig: Der Hash wird **auf dem Server** gebildet. In der Datenbank liegt nur der Hash. Selbst die Administratoren sehen das Passwort niemals im Klartext. Moderne Frameworks wie Django, Rails oder Spring Security erledigen diesen Ablauf automatisch – du musst ihn in der Praxis selten selbst implementieren. Verstehen solltest du ihn trotzdem, damit du bei der Architektur eigener Systeme die richtigen Fragen stellen kannst.

## 5. Angriffe auf Passwort-Hashes

Die Einweg-Eigenschaft schützt **nicht** vor allen Angriffen. Ein Angreifer kann die Rechnung umdrehen: Statt aus dem Hash auf das Passwort zu schließen, **berät er sich selbst viele Kandidat-Passwörter zusammen** und hasht sie durch. Passt einer der berechneten Hashes zum gestohlenen Hash, hat er das Passwort gefunden. Dieser Ansatz ist besonders effektiv, weil Menschen extrem schlechte Passwörter wählen: `123456`, `passwort`, `qwerty`, `admin` oder der Vorname des Haustiers.

### 5.1 Wörterbuch-Angriff

Beim **Wörterbuch-Angriff** nimmt der Angreifer eine große Liste häufiger Passwörter – oft mehrere Millionen Einträge, zum Beispiel die berühmte `rockyou.txt` aus einem alten Datenleck – und hasht jeden Eintrag. Jeder berechnete Hash wird mit dem gestohlenen Hash verglichen. Moderne GPUs schaffen Milliarden SHA-256-Berechnungen pro Sekunde, einfache Passwörter sind damit in wenigen Sekunden geknackt.

In [4]:
from IPython.display import Markdown, display
with open("diagramme/03_wortbuch_angriff.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
flowchart TD
    Start(["Start: ziel_hash bekannt"]) --> L["Lade Wortliste<br/>z.B. rockyou.txt"]
    L --> F{{"Fuer jedes Wort<br/>in Wortliste"}}
    F --> H["h = sha256(wort)"]
    H --> V{"h == ziel_hash?"}
    V -- "Ja" --> G(["Gefunden:<br/>Passwort = wort"])
    V -- "Nein" --> F
    F -- "Ende der Liste" --> N(["Nicht gefunden"])

```

### 5.2 Rainbow-Tables

**Rainbow-Tables** sind vorausberechnete Tabellen, die Millionen von Hashes fertig auf der Festplatte ablegen. Statt bei jedem Angriff neu zu rechnen, schlägt man den gestohlenen Hash einfach in der Tabelle nach. Der Angreifer tauscht Rechenzeit gegen Speicherplatz und ist bei einem Angriff extrem schnell.

### 5.3 Die Gegenmaßnahme: **Salt**

Ein **Salt** ist ein zufälliger Wert, den der Server bei der Registrierung pro Nutzer erzeugt und zusammen mit dem Hash speichert. Beim Login wird nicht mehr nur das Passwort gehasht, sondern die Kombination `passwort + salt`. Der gespeicherte Datensatz sieht dann etwa so aus: `(email, salt, hash_von(passwort + salt))`.

Das hat zwei Effekte: Erstens wird eine vorgerechnete Rainbow-Table wertlos, denn der Angreifer müsste für jeden Salt eine eigene Tabelle anlegen. Zweitens haben zwei Nutzer mit dem gleichen Passwort trotzdem unterschiedliche Hashes, denn ihr Salt ist unterschiedlich. Ein Angriff muss also immer **pro Nutzer einzeln** laufen, was den Aufwand massiv erhöht.

> [!NOTE]
> Ein **Salt** ist ein zufälliger Zusatzwert, der vor dem Hashen an das Passwort angehängt wird. Er wird unverschlüsselt mitgespeichert – sein Nutzen liegt nicht in der Geheimhaltung, sondern darin, dass jeder Datensatz einen **individuellen** Hash erzeugt.

### 5.4 Langsame Passwort-Hashes: bcrypt, scrypt, Argon2

SHA-256 ist **zu schnell** für Passwort-Hashing. Genau diese Eigenschaft, die bei der Datei-Integritätsprüfung ein Vorteil ist, hilft dem Angreifer. Deshalb wurden spezielle **Passwort-Hash-Funktionen** entwickelt, die absichtlich langsam sind und einen einstellbaren Kostenfaktor haben: **bcrypt** (1999), **scrypt** (2009) und **Argon2** (2015, Sieger der *Password Hashing Competition*). In Python nutzt du dafür die Pakete `bcrypt` oder `argon2-cffi`. In dieser Vorlesung bleiben wir bei SHA-256, damit du die Grundprinzipien ohne zusätzliche Bibliothek verstehst. In echter Produktions-Software solltest du immer einen spezialisierten Passwort-Hasher verwenden.

## 6. Kollisionen – wenn zwei Eingaben denselben Hash erzeugen

Da eine Hash-Funktion aus beliebig langen Eingaben einen fest langen Output macht, muss es logischerweise **Kollisionen** geben: zwei verschiedene Eingaben mit demselben Hash-Wert. Bei SHA-256 gibt es 2²⁵⁶ mögliche Hash-Werte – eine astronomisch große, aber endliche Zahl. Die entscheidende Frage lautet also nicht „Gibt es Kollisionen?" sondern „Wie schwer ist es, eine Kollision **gezielt** zu finden?".

Das **Geburtstagsparadoxon** aus der Wahrscheinlichkeitstheorie sagt: Wenn du 23 Personen in einen Raum steckst, ist die Wahrscheinlichkeit, dass zwei von ihnen am selben Tag Geburtstag haben, schon über 50 %. Übertragen auf Hash-Funktionen heißt das: Mit etwa 2^(n/2) zufälligen Versuchen findest du eine Kollision in einem n-Bit-Hash. Bei SHA-256 sind das 2¹²⁸ Versuche – immer noch jenseits aller technischen Möglichkeiten. Bei MD5 sind es nur 2⁶⁴, und genau dieser Angriff ist 2004 gelungen.

## 7. Integritätsprüfung von Dateien

Bevor wir zur digitalen Signatur kommen, noch ein einfacherer Anwendungsfall: die **Integritätsprüfung**. Wenn du eine große Datei über das Internet herunterlädst, willst du sicherstellen, dass sie unterwegs nicht beschädigt wurde – weder durch Übertragungsfehler noch durch einen Angreifer, der die Datei manipuliert hat. Der Hersteller veröffentlicht dazu zusammen mit der Datei einen SHA-256-Hash. Nach dem Download rechnest du lokal den Hash und vergleichst ihn.

Eine **Integritätsprüfung** schützt nur gegen zufällige Fehler, nicht gegen einen Angreifer. Wenn derselbe Angreifer sowohl die Datei als auch den publizierten Hash manipulieren kann, merkst du nichts. Deshalb braucht es für echte Sicherheit eine **signierte** Angabe des Hashes – und damit sind wir bei digitalen Signaturen.

## 8. Digitale Signaturen

Erinnere dich an V17: Bei der **asymmetrischen Kryptografie** hat jeder Teilnehmer ein Schlüsselpaar – einen **privaten Schlüssel** (geheim, nur der Besitzer kennt ihn) und einen **öffentlichen Schlüssel** (darf jeder kennen). Daten, die mit dem privaten Schlüssel verschlüsselt wurden, lassen sich nur mit dem passenden öffentlichen Schlüssel wieder entschlüsseln. Diese Asymmetrie nutzt die **digitale Signatur**.

### 8.1 Signatur erstellen

Der Hersteller einer Firmware möchte einem Kunden beweisen, dass die Datei tatsächlich von ihm stammt und nicht verändert wurde. Er geht in zwei Schritten vor:

1. Er berechnet den **SHA-256-Hash** der Firmware-Datei. Das Ergebnis ist ein kompakter Fingerabdruck der Datei.
2. Er **verschlüsselt diesen Hash mit seinem privaten Schlüssel**. Das Ergebnis ist die **digitale Signatur** und wird zusammen mit der Firmware veröffentlicht.

### 8.2 Signatur prüfen

Der Empfänger – zum Beispiel der Roboterarm, der das Update installieren soll – geht die beiden Schritte rückwärts:

1. Er berechnet selbst den **SHA-256-Hash** der empfangenen Firmware-Datei.
2. Er **entschlüsselt die Signatur mit dem öffentlichen Schlüssel des Herstellers**. Heraus kommt der Hash, den der Hersteller damals berechnet hat.
3. Er **vergleicht** beide Hashes. Sind sie identisch, ist die Firmware unverändert **und** tatsächlich vom Hersteller. Unterscheiden sich die Hashes, wird das Update **abgelehnt**.

In [5]:
from IPython.display import Markdown, display
with open("diagramme/04_digitale_signatur.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
flowchart LR
    subgraph Erstellung["Erstellung (Hersteller)"]
        F1["Firmware.bin"] --> H1["SHA-256 Hash"]
        H1 --> E1["Verschluesseln mit<br/>privatem Schluessel"]
        E1 --> S1["Signatur"]
    end
    subgraph Pruefung["Pruefung (Roboter)"]
        F2["Firmware.bin"] --> H2["SHA-256 Hash"]
        S2["Signatur"] --> D2["Entschluesseln mit<br/>oeffentlichem Schluessel"]
        D2 --> V{"Hashes gleich?"}
        H2 --> V
        V -- "Ja" --> OK(["Update installieren"])
        V -- "Nein" --> X(["Update ablehnen"])
    end
    S1 -. uebertragen .-> S2
    F1 -. uebertragen .-> F2

```

### 8.3 Warum überhaupt der Umweg über den Hash?

Theoretisch könnte der Hersteller die komplette Firmware direkt mit seinem privaten Schlüssel verschlüsseln und das Ergebnis als Signatur mitsenden. In der Praxis scheitert das an zwei Punkten: Asymmetrische Verschlüsselung ist **sehr langsam** und nur für kleine Datenmengen praktikabel, und eine Firmware hat oft mehrere Megabyte. Der Trick mit dem Hash reduziert das Signieren auf 256 Bit – das geht in Millisekunden.

### 8.4 Wer garantiert den öffentlichen Schlüssel?

Die ganze Konstruktion hängt an einem Faden: Der Empfänger muss sich sicher sein, dass der **öffentliche Schlüssel** tatsächlich zum Hersteller gehört. Ein Angreifer könnte sonst einen eigenen Schlüssel als „Herstellerschlüssel" verbreiten und selbst gültige Signaturen ausstellen. Dieses Problem löst eine **Certificate Authority (CA)** – eine vertrauenswürdige Instanz, die Zertifikate ausstellt und ihrerseits wieder signiert. Die Details (Zertifikatskette, Root CAs) schauen wir uns in V19/V20 im Kontext von TLS nicht erneut an, der wichtige Punkt für heute ist: In der Praxis wird nie ein nackter öffentlicher Schlüssel ausgetauscht, sondern ein **Zertifikat**, das den Schlüssel an eine Identität bindet.

## 9. HMAC – Hash mit Schlüssel

Zwischen reinem Hash und voller digitaler Signatur gibt es noch einen Mittelweg: **HMAC** (*Hash-based Message Authentication Code*). Dabei wird der Hash nicht nur aus der Nachricht gebildet, sondern aus der Kombination **Nachricht + gemeinsamem Geheimschlüssel**. Zwei Parteien, die den Schlüssel vorab ausgetauscht haben, können so die Echtheit einer Nachricht prüfen, ohne asymmetrische Kryptografie zu brauchen. HMAC ist schnell, leicht zu implementieren und in vielen Protokollen (z. B. JWT, TLS, API-Signaturen wie AWS) im Einsatz.

## 10. Maschinenbau-Beispiel: Firmware-Update für einen Roboterarm

Ein Industrie-Roboter bekommt regelmäßig **Over-The-Air-Updates** für seine Firmware. Das Szenario ist sicherheitskritisch: Eine manipulierte Firmware könnte den Roboter dazu bringen, unkontrollierte Bewegungen auszuführen, wodurch Menschen verletzt oder Maschinen beschädigt werden. Genau deshalb prüft die Steuerungseinheit **vor** jeder Installation die digitale Signatur des Herstellers.

Der typische Ablauf:

1. Der Hersteller veröffentlicht eine neue Firmware `firmware_v2.3.0.bin` zusammen mit einer JSON-Datei, die unter anderem den **SHA-256-Hash** der Firmware und eine **digitale Signatur** enthält. Ein solcher Datensatz liegt im Ordner `testdaten/` bereit und sieht in verkürzter Form etwa so aus: `{"sha256_hash": "...", "signature": "...", "signed_by": "RobotCorp CA"}`.
2. Der Roboter lädt beide Dateien herunter.
3. Er berechnet selbst den SHA-256-Hash der `.bin`-Datei und vergleicht ihn mit dem Feld `sha256_hash`.
4. Er prüft die Signatur mit dem öffentlichen Schlüssel der Herstellers-CA.
5. Nur wenn beide Prüfungen erfolgreich sind, wird das Update installiert und anschließend neu gestartet.

Im Praxis-Notebook bauen wir Schritt 3 konkret nach – die Signaturprüfung mit echten RSA-Schlüsseln sparen wir uns, sie läuft konzeptionell aber genauso.

## 11. Selbst-Check

Bevor du weitermachst, beantworte die folgenden Fragen. Klicke erst danach auf das Dreieck, um die Antwort aufzuklappen.

<details><summary>❓ <strong>1. Was liefert eine Hash-Funktion?</strong></summary>

Einen **fest langen** Wert (z. B. 256 Bit bei SHA-256), der aus einer beliebig großen Eingabe berechnet wird. Er ist deterministisch, aber praktisch nicht umkehrbar.

</details>

<details><summary>❓ <strong>2. Warum darf MD5 nicht mehr für Signaturen eingesetzt werden?</strong></summary>

Weil seit 2004 effiziente **Kollisionsangriffe** bekannt sind. Ein Angreifer kann zwei Dokumente erzeugen, die zwar unterschiedlichen Inhalt haben, aber denselben MD5-Hash – und damit wäre jede auf MD5 aufbauende Signatur austauschbar.

</details>

<details><summary>❓ <strong>3. Warum speichert ein Server Passwörter als Hash und nicht im Klartext?</strong></summary>

Damit ein Datenbank-Diebstahl den Angreifer nicht sofort in den Besitz aller Passwörter bringt. Die Einweg-Eigenschaft des Hashes macht es praktisch unmöglich, aus dem gestohlenen Hash das Klartext-Passwort zu rekonstruieren.

</details>

<details><summary>❓ <strong>4. Wie funktioniert ein Wörterbuch-Angriff?</strong></summary>

Der Angreifer nimmt eine Liste häufig verwendeter Passwörter, hasht jedes davon und vergleicht das Ergebnis mit dem gestohlenen Hash. Ist ein Passwort einfach oder in der Liste enthalten, wird es in Sekunden geknackt.

</details>

<details><summary>❓ <strong>5. Was bringt ein Salt?</strong></summary>

Ein Salt ist ein zufälliger Zusatzwert pro Nutzer, der vor dem Hashen an das Passwort angehängt wird. Dadurch werden **vorgerechnete Rainbow-Tables wertlos**, und zwei Nutzer mit identischem Passwort bekommen trotzdem unterschiedliche Hashes.

</details>

<details><summary>❓ <strong>6. Welche beiden Werte vergleicht der Roboter bei der Firmware-Prüfung?</strong></summary>

Den **selbst berechneten** SHA-256-Hash der heruntergeladenen `.bin`-Datei und den Hash, den er aus der mitgelieferten Signatur nach Entschlüsselung mit dem öffentlichen Schlüssel herausbekommt. Nur bei Gleichheit wird das Update installiert.

</details>

<details><summary>❓ <strong>7. Warum reicht es nicht, eine Datei mit ihrem MD5-Hash als „sicher" zu betrachten, wenn beides vom selben Server kommt?</strong></summary>

Weil ein Angreifer, der den Server kontrolliert, sowohl die Datei als auch den publizierten Hash austauschen kann. Der MD5-Hash schützt nur gegen **zufällige** Übertragungsfehler, nicht gegen einen böswilligen Akteur. Für echten Schutz brauchst du eine **Signatur**, die mit einem unabhängig vertrauenswürdigen Schlüssel erstellt wurde.

</details>

<details><summary>❓ <strong>8. Warum wird beim Signieren zuerst gehasht und dann verschlüsselt, statt direkt die ganze Datei zu verschlüsseln?</strong></summary>

Weil asymmetrische Verschlüsselung sehr langsam ist und bei einer mehrere MB großen Firmware unpraktikabel wäre. Der Hash reduziert die Eingabe auf 256 Bit, und nur dieser kompakte Fingerabdruck muss mit dem privaten Schlüssel verschlüsselt werden.

</details>

## ✅ Zusammenfassung

- Eine **Hash-Funktion** erzeugt aus beliebig langen Eingaben einen fest langen, deterministischen, einweg-gerichteten Fingerabdruck mit starkem **Lawineneffekt**.
- **MD5** und **SHA-1** gelten als gebrochen und dürfen nicht mehr für Sicherheit eingesetzt werden. **SHA-256** ist der aktuelle Standard.
- **Passwörter** werden niemals im Klartext gespeichert, sondern als Hash – idealerweise mit **Salt** und einer langsamen Passwort-Hash-Funktion (bcrypt, Argon2).
- Ein **Wörterbuch-Angriff** probiert eine Liste von Kandidaten durch und funktioniert nur gegen schwache Passwörter.
- Eine **digitale Signatur** = Hash(Datei) + asymmetrische Verschlüsselung mit dem privaten Schlüssel. Empfänger prüfen sie mit dem öffentlichen Schlüssel – Standardverfahren für Firmware-Updates.

## ➡️ Nächster Schritt
Weiter mit [02_praxis.ipynb](02_praxis.ipynb) – dort lernst du `hashlib` in Python praktisch kennen.